# Classifier Model EDA

This notebook evaluates a trained cell-quality classifier on crop data and answers:
- How well does it separate `good` vs `bad` cells?
- Is the task currently doable under strict thresholds?

Ground truth is read from crop folder labels (`good`/`bad`) under:
`classifier_output/crops/<mask_bg_mode>/<split>/`.

## 1) Configuration

Edit these values before running `Restart & Run All`.

In [ ]:
from pathlib import Path

# Required inputs
CHECKPOINT_PATH = Path("../classifier_output/clf_resnet18_freezefalse_lr0.0001_thr0.7_maskbgfalse_cv0_20260319_225955/best_model.pt")
CROPS_ROOT = Path("../classifier_output/crops")
MASK_BG_MODE = "mask_bg_false"  # mask_bg_true | mask_bg_false
EVAL_SPLIT = "test"             # preferred split
ENABLE_SPLIT_FALLBACK = True     # test -> val -> train

# Decision rule
DECISION_THRESHOLD = 0.5
PASS_F1 = 0.80
PASS_SPECIFICITY = 0.80

# Runtime
BATCH_SIZE = 128
NUM_WORKERS = 4
N_BOOTSTRAP = 1000
RANDOM_SEED = 42

# Visualization sampling
N_EXAMPLES_PER_GROUP = 8
N_LOW_CONF_EXAMPLES = 12
N_GRADCAM_EXAMPLES = 6
MAX_TSNE_POINTS = 2000

## 2) Imports and Environment Setup

In [ ]:
from __future__ import annotations

import random
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from IPython.display import Markdown, display
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (
    accuracy_score,
    auc,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_curve,
)
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src" / "cell_size").is_dir():
            return p
    raise FileNotFoundError("Could not locate repo root containing src/cell_size")


set_seed(RANDOM_SEED)
REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from cell_size.classifier.dataset import IMAGENET_MEAN, IMAGENET_STD
from cell_size.classifier.models import build_model

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Repo root: {REPO_ROOT}")
print(f"Device: {DEVICE}")

## 3) Helper Functions

In [ ]:
class ImageFolderWithPaths(ImageFolder):
    def __getitem__(self, index):
        image, target = super().__getitem__(index)
        path, _ = self.samples[index]
        return image, target, path


def resolve_checkpoint(path: Path, search_root: Path) -> Path:
    p = Path(path)
    if p.is_file():
        return p.resolve()

    candidates = sorted(
        search_root.rglob("best_model.pt"),
        key=lambda x: x.stat().st_mtime,
        reverse=True,
    )
    if candidates:
        print(f"CHECKPOINT_PATH not found; using newest discovered checkpoint: {candidates[0]}")
        return candidates[0].resolve()

    raise FileNotFoundError(
        f"Checkpoint not found at {p} and no best_model.pt found under {search_root}."
    )


def resolve_split_dir(crops_root: Path, mask_bg_mode: str, preferred_split: str, fallback: bool) -> tuple[Path, str, list[str]]:
    base = crops_root / mask_bg_mode
    if not base.is_dir():
        raise FileNotFoundError(f"Mask mode directory not found: {base}")

    order = [preferred_split]
    if fallback:
        if preferred_split == "test":
            order.extend(["val", "train"])
        elif preferred_split == "val":
            order.append("train")

    seen = set()
    order = [s for s in order if not (s in seen or seen.add(s))]

    for split in order:
        split_dir = base / split
        if (split_dir / "good").is_dir() and (split_dir / "bad").is_dir():
            return split_dir, split, order

    raise FileNotFoundError(
        f"No usable split found in {base}. Checked order: {order}"
    )


def build_eval_transform(crop_size: int) -> transforms.Compose:
    return transforms.Compose([
        transforms.Resize((crop_size, crop_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def to_binary_targets(targets: np.ndarray, good_idx: int) -> np.ndarray:
    return (targets == good_idx).astype(int)


def confusion_case(y_true: int, y_pred: int) -> str:
    if y_true == 1 and y_pred == 1:
        return "TP"
    if y_true == 0 and y_pred == 0:
        return "TN"
    if y_true == 0 and y_pred == 1:
        return "FP"
    return "FN"


def compute_binary_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "specificity": float(specificity),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "support_bad": int((y_true == 0).sum()),
        "support_good": int((y_true == 1).sum()),
    }


def bootstrap_metric_ci(
    y_true: np.ndarray,
    probs: np.ndarray,
    threshold: float,
    n_bootstrap: int,
    seed: int,
) -> tuple[dict[str, tuple[float, float]], pd.DataFrame]:
    rng = np.random.default_rng(seed)
    n = len(y_true)
    rows = []

    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, n)
        yt = y_true[idx]
        yp = (probs[idx] >= threshold).astype(int)
        rows.append(compute_binary_metrics(yt, yp))

    boot_df = pd.DataFrame(rows)
    ci = {}
    for metric in ["accuracy", "recall", "precision", "f1", "specificity"]:
        vals = boot_df[metric].astype(float).to_numpy()
        vals = vals[np.isfinite(vals)]
        if len(vals) == 0:
            ci[metric] = (np.nan, np.nan)
        else:
            ci[metric] = (float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5)))

    return ci, boot_df


def threshold_sweep(y_true: np.ndarray, probs: np.ndarray, thresholds: np.ndarray) -> pd.DataFrame:
    rows = []
    for t in thresholds:
        yp = (probs >= t).astype(int)
        m = compute_binary_metrics(y_true, yp)
        rows.append({
            "threshold": float(t),
            "accuracy": m["accuracy"],
            "recall": m["recall"],
            "precision": m["precision"],
            "f1": m["f1"],
            "specificity": m["specificity"],
        })
    return pd.DataFrame(rows)


def load_rgb(path: str | Path) -> np.ndarray:
    with Image.open(path) as img:
        return np.array(img.convert("RGB"))


def plot_image_grid(paths: list[str], titles: list[str], suptitle: str, ncols: int = 4, figsize_scale: float = 3.0) -> None:
    if len(paths) == 0:
        print(f"No examples available for: {suptitle}")
        return

    n = len(paths)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_scale * ncols, figsize_scale * nrows))
    axes = np.atleast_1d(axes).reshape(nrows, ncols)

    for i in range(nrows * ncols):
        ax = axes[i // ncols, i % ncols]
        ax.axis("off")
        if i >= n:
            continue
        img = load_rgb(paths[i])
        ax.imshow(img)
        ax.set_title(titles[i], fontsize=9)

    fig.suptitle(suptitle, fontsize=13)
    plt.tight_layout()
    plt.show()


def sample_rows(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    if len(df) == 0:
        return df
    return df.sample(n=min(n, len(df)), random_state=seed)


def get_head_module(model: nn.Module, encoder: str) -> nn.Module:
    if encoder.startswith("resnet"):
        return model.fc
    if encoder == "efficientnet_b0":
        return model.classifier[1]
    if encoder == "vit_b_16":
        return model.heads.head
    if encoder == "squeezenet1_1":
        return model.classifier[2]
    raise ValueError(f"Unsupported encoder for head hook: {encoder}")


def extract_penultimate_features(
    model: nn.Module,
    encoder: str,
    loader: DataLoader,
    device: torch.device,
) -> tuple[list[str], np.ndarray]:
    features = []
    paths_all = []
    cache: dict[str, torch.Tensor] = {}

    head = get_head_module(model, encoder)

    def _hook(module, inputs, output):
        cache["feat"] = inputs[0].detach().cpu()

    handle = head.register_forward_hook(_hook)
    model.eval()
    with torch.no_grad():
        for images, _, paths in loader:
            logits = model(images.to(device))
            _ = logits  # ensure forward runs for hook
            feat = cache["feat"].numpy()
            features.append(feat)
            paths_all.extend(paths)

    handle.remove()

    if not features:
        return [], np.empty((0, 1), dtype=np.float32)
    return paths_all, np.vstack(features)


def last_conv_layer(model: nn.Module) -> tuple[str, nn.Module | None]:
    name_out = ""
    layer_out = None
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            name_out = name
            layer_out = module
    return name_out, layer_out


def compute_gradcam(
    model: nn.Module,
    input_tensor: torch.Tensor,
    target_layer: nn.Module,
    class_index: int,
) -> np.ndarray:
    activations = []
    gradients = []

    def fwd_hook(module, inputs, output):
        activations.append(output.detach())

    def bwd_hook(module, grad_input, grad_output):
        gradients.append(grad_output[0].detach())

    h1 = target_layer.register_forward_hook(fwd_hook)
    h2 = target_layer.register_full_backward_hook(bwd_hook)

    model.zero_grad(set_to_none=True)
    logits = model(input_tensor)
    score = logits[:, 0] if class_index == 1 else -logits[:, 0]
    score.backward(torch.ones_like(score))

    h1.remove()
    h2.remove()

    if not activations or not gradients:
        raise RuntimeError("Failed to collect activations/gradients for Grad-CAM")

    acts = activations[-1]
    grads = gradients[-1]
    weights = grads.mean(dim=(2, 3), keepdim=True)
    cam = (weights * acts).sum(dim=1, keepdim=True)
    cam = F.relu(cam)
    cam = F.interpolate(cam, size=input_tensor.shape[-2:], mode="bilinear", align_corners=False)
    cam = cam[0, 0].cpu().numpy()

    cam_min, cam_max = float(cam.min()), float(cam.max())
    if cam_max > cam_min:
        cam = (cam - cam_min) / (cam_max - cam_min)
    else:
        cam = np.zeros_like(cam)
    return cam


def overlay_cam(rgb: np.ndarray, cam: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    base = rgb.astype(np.float32) / 255.0
    heat = plt.cm.jet(cam)[..., :3].astype(np.float32)
    over = (1 - alpha) * base + alpha * heat
    return np.clip(over, 0.0, 1.0)

## 4) Load Checkpoint, Resolve Split, Run Fresh Inference

In [ ]:
checkpoint_path = resolve_checkpoint(Path(CHECKPOINT_PATH), REPO_ROOT / "classifier_output")
split_dir, resolved_split, checked_order = resolve_split_dir(
    Path(CROPS_ROOT), MASK_BG_MODE, EVAL_SPLIT, ENABLE_SPLIT_FALLBACK
)

ckpt = torch.load(str(checkpoint_path), map_location=DEVICE, weights_only=False)
encoder = ckpt["encoder"]
crop_size = int(ckpt.get("crop_size", 224))

model = build_model(encoder=encoder, pretrained=False, freeze_encoder=False)
model.load_state_dict(ckpt["model_state_dict"])
model.to(DEVICE)
model.eval()

print(f"Checkpoint: {checkpoint_path}")
print(f"Encoder: {encoder}")
print(f"Crop size from checkpoint: {crop_size}")
print(f"Requested split: {EVAL_SPLIT}")
print(f"Fallback order considered: {checked_order}")
print(f"Resolved split: {resolved_split}")
print(f"Split directory: {split_dir}")

eval_transform = build_eval_transform(crop_size)
eval_ds = ImageFolderWithPaths(str(split_dir), transform=eval_transform)
print(f"Class to idx: {eval_ds.class_to_idx}")

if "good" not in eval_ds.class_to_idx or "bad" not in eval_ds.class_to_idx:
    raise ValueError(f"Expected classes 'good' and 'bad', got: {eval_ds.class_to_idx}")

good_idx = eval_ds.class_to_idx["good"]

loader = DataLoader(
    eval_ds,
    batch_size=int(BATCH_SIZE),
    shuffle=False,
    num_workers=int(NUM_WORKERS),
    pin_memory=torch.cuda.is_available(),
)

rows = []
with torch.no_grad():
    for images, targets, paths in loader:
        logits = model(images.to(DEVICE))
        probs_good = torch.sigmoid(logits).squeeze(1).cpu().numpy()
        y_true = to_binary_targets(targets.numpy(), good_idx)
        y_pred = (probs_good >= DECISION_THRESHOLD).astype(int)

        for p, yt, yp, prob in zip(paths, y_true, y_pred, probs_good):
            rows.append({
                "path": str(p),
                "split": resolved_split,
                "y_true": int(yt),
                "y_pred": int(yp),
                "p_good": float(prob),
                "p_bad": float(1.0 - prob),
                "true_label": "good" if int(yt) == 1 else "bad",
                "pred_label": "good" if int(yp) == 1 else "bad",
                "correct": bool(int(yt) == int(yp)),
                "case": confusion_case(int(yt), int(yp)),
            })

result_df = pd.DataFrame(rows)
if result_df.empty:
    raise RuntimeError("No samples were evaluated. Check split directories and crop files.")

print(f"Evaluated samples: {len(result_df)}")
result_df.head()

## 5) Core Metrics at Decision Threshold

In [ ]:
y_true = result_df["y_true"].to_numpy(dtype=int)
y_pred = result_df["y_pred"].to_numpy(dtype=int)
probs = result_df["p_good"].to_numpy(dtype=float)

metrics = compute_binary_metrics(y_true, y_pred)
metrics_df = pd.DataFrame([
    {
        "accuracy": metrics["accuracy"],
        "recall": metrics["recall"],
        "precision": metrics["precision"],
        "f1": metrics["f1"],
        "specificity": metrics["specificity"],
        "support_good": metrics["support_good"],
        "support_bad": metrics["support_bad"],
        "threshold": DECISION_THRESHOLD,
    }
])

display(metrics_df.round(4))

## 6) Confusion Matrix (Counts + Row-Normalized)

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
cm_norm = cm.astype(float) / np.clip(cm.sum(axis=1, keepdims=True), 1, None)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, arr, title, fmt in [
    (axes[0], cm, "Confusion Matrix (Counts)", "d"),
    (axes[1], cm_norm, "Confusion Matrix (Row-Normalized)", ".2f"),
]:
    im = ax.imshow(arr, cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks([0, 1], labels=["bad", "good"])
    ax.set_yticks([0, 1], labels=["bad", "good"])
    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            txt = format(arr[i, j], fmt)
            ax.text(j, i, txt, ha="center", va="center", color="black")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

# Metric correctness check: specificity from confusion matrix
spec_from_cm = cm[0, 0] / (cm[0, 0] + cm[0, 1]) if (cm[0, 0] + cm[0, 1]) > 0 else np.nan
assert np.isclose(spec_from_cm, metrics["specificity"], equal_nan=True), (
    f"Specificity mismatch: cm={spec_from_cm}, metrics={metrics['specificity']}"
)
print("Specificity check passed.")

## 7) Threshold Sweep + ROC/PR Curves

In [ ]:
thresholds = np.linspace(0.0, 1.0, 101)
sweep_df = threshold_sweep(y_true, probs, thresholds)

assert float(sweep_df["threshold"].iloc[0]) == 0.0
assert float(sweep_df["threshold"].iloc[-1]) == 1.0

fig, ax = plt.subplots(figsize=(10, 6))
for col in ["accuracy", "recall", "precision", "f1", "specificity"]:
    ax.plot(sweep_df["threshold"], sweep_df[col], label=col)
ax.axvline(DECISION_THRESHOLD, color="black", linestyle="--", linewidth=1.2, label="decision_threshold")
ax.set_xlabel("Threshold for classifying as good")
ax.set_ylabel("Metric")
ax.set_title("Threshold Sweep")
ax.set_ylim(0, 1.02)
ax.grid(alpha=0.3)
ax.legend(loc="best")
plt.show()

if len(np.unique(y_true)) < 2:
    warnings.warn("Only one class present; ROC/PR curves are undefined.")
    roc_auc = np.nan
    pr_auc = np.nan
else:
    fpr, tpr, _ = roc_curve(y_true, probs)
    roc_auc = auc(fpr, tpr)

    pr, rc, _ = precision_recall_curve(y_true, probs)
    pr_auc = auc(rc, pr)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].plot(fpr, tpr, label=f"ROC AUC={roc_auc:.4f}")
    axes[0].plot([0, 1], [0, 1], "k--", linewidth=1)
    axes[0].set_xlabel("False Positive Rate")
    axes[0].set_ylabel("True Positive Rate")
    axes[0].set_title("ROC Curve")
    axes[0].grid(alpha=0.3)
    axes[0].legend(loc="lower right")

    axes[1].plot(rc, pr, label=f"PR AUC={pr_auc:.4f}")
    baseline = float((y_true == 1).mean())
    axes[1].hlines(baseline, 0, 1, colors="k", linestyles="--", linewidth=1, label=f"baseline={baseline:.3f}")
    axes[1].set_xlabel("Recall")
    axes[1].set_ylabel("Precision")
    axes[1].set_title("Precision-Recall Curve")
    axes[1].grid(alpha=0.3)
    axes[1].legend(loc="best")

    plt.tight_layout()
    plt.show()

print(f"ROC AUC: {roc_auc:.4f}" if np.isfinite(roc_auc) else "ROC AUC: NaN")
print(f"PR AUC:  {pr_auc:.4f}" if np.isfinite(pr_auc) else "PR AUC: NaN")

## 8) 95% Bootstrap Confidence Intervals

In [ ]:
ci_dict, boot_df = bootstrap_metric_ci(
    y_true=y_true,
    probs=probs,
    threshold=DECISION_THRESHOLD,
    n_bootstrap=int(N_BOOTSTRAP),
    seed=int(RANDOM_SEED),
)

ci_rows = []
for metric_name in ["accuracy", "recall", "precision", "f1", "specificity"]:
    lo, hi = ci_dict[metric_name]
    ci_rows.append({
        "metric": metric_name,
        "point_estimate": metrics[metric_name],
        "ci_2.5%": lo,
        "ci_97.5%": hi,
    })
ci_df = pd.DataFrame(ci_rows)
display(ci_df.round(4))

## 9) Class Balance and Confidence Distributions

In [ ]:
class_counts = result_df["true_label"].value_counts().rename_axis("true_label").reset_index(name="count")
display(class_counts)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(class_counts["true_label"], class_counts["count"], color=["tab:orange", "tab:green"])
axes[0].set_title("Class Balance (Ground Truth)")
axes[0].set_ylabel("Count")

bad_probs = result_df.loc[result_df["y_true"] == 0, "p_good"].to_numpy()
good_probs = result_df.loc[result_df["y_true"] == 1, "p_good"].to_numpy()

axes[1].hist(bad_probs, bins=25, alpha=0.7, label="true bad", color="tab:orange")
axes[1].hist(good_probs, bins=25, alpha=0.7, label="true good", color="tab:green")
axes[1].axvline(DECISION_THRESHOLD, color="black", linestyle="--", linewidth=1.2)
axes[1].set_title("Predicted P(good) Distribution by True Class")
axes[1].set_xlabel("P(good)")
axes[1].set_ylabel("Count")
axes[1].legend(loc="best")

plt.tight_layout()
plt.show()

## 10) Qualitative Examples

Panels shown:
- Random true `good` examples
- Random true `bad` examples
- TP / TN / FP / FN examples
- Low-confidence examples near threshold

In [ ]:
def show_panel(df: pd.DataFrame, title: str, n: int, seed: int) -> None:
    sub = sample_rows(df, n, seed)
    paths = sub["path"].tolist()
    titles = [
        f"{r.true_label}->{r.pred_label} p={r.p_good:.2f}" for r in sub.itertuples(index=False)
    ]
    plot_image_grid(paths, titles, suptitle=title, ncols=4)

show_panel(result_df[result_df["y_true"] == 1], "Random True Good Cells", N_EXAMPLES_PER_GROUP, RANDOM_SEED)
show_panel(result_df[result_df["y_true"] == 0], "Random True Bad Cells", N_EXAMPLES_PER_GROUP, RANDOM_SEED + 1)

for case_name in ["TP", "TN", "FP", "FN"]:
    show_panel(
        result_df[result_df["case"] == case_name],
        f"{case_name} Examples",
        N_EXAMPLES_PER_GROUP,
        RANDOM_SEED + hash(case_name) % 1000,
    )

low_conf_df = result_df.assign(distance_to_threshold=(result_df["p_good"] - DECISION_THRESHOLD).abs())
low_conf_df = low_conf_df.sort_values("distance_to_threshold", ascending=True).head(N_LOW_CONF_EXAMPLES)
show_panel(low_conf_df, "Low-Confidence Samples Near Threshold", N_LOW_CONF_EXAMPLES, RANDOM_SEED)

## 11) Separability (Penultimate Features + PCA + t-SNE)

In [ ]:
feat_paths, feat_matrix = extract_penultimate_features(model, encoder, loader, DEVICE)
if len(feat_paths) != len(result_df):
    raise RuntimeError(f"Feature count mismatch: features={len(feat_paths)} rows={len(result_df)}")

# Align features to result rows by path.
feat_lookup = {p: f for p, f in zip(feat_paths, feat_matrix)}
X = np.vstack([feat_lookup[p] for p in result_df["path"].tolist()])

viz_df = result_df.copy()
if len(viz_df) > MAX_TSNE_POINTS:
    viz_df = (
        viz_df.groupby("y_true", group_keys=False)
        .apply(lambda x: x.sample(n=min(len(x), MAX_TSNE_POINTS // 2), random_state=RANDOM_SEED))
        .reset_index(drop=True)
    )

X_viz = np.vstack([feat_lookup[p] for p in viz_df["path"].tolist()])

if X_viz.shape[0] < 5:
    print("Too few samples for robust t-SNE visualization.")
else:
    n_pca = min(50, X_viz.shape[1], X_viz.shape[0] - 1)
    if n_pca < 2:
        n_pca = min(2, X_viz.shape[1])

    X_pca = PCA(n_components=n_pca, random_state=RANDOM_SEED).fit_transform(X_viz)

    perplexity = min(30, max(5, X_pca.shape[0] // 10))
    perplexity = min(perplexity, X_pca.shape[0] - 1)

    X_tsne = TSNE(
        n_components=2,
        random_state=RANDOM_SEED,
        init="pca",
        learning_rate="auto",
        perplexity=perplexity,
    ).fit_transform(X_pca)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    color_true = np.where(viz_df["y_true"].to_numpy() == 1, "tab:green", "tab:orange")
    axes[0].scatter(X_tsne[:, 0], X_tsne[:, 1], c=color_true, s=16, alpha=0.8)
    axes[0].set_title("t-SNE Colored by True Class")
    axes[0].set_xlabel("t-SNE 1")
    axes[0].set_ylabel("t-SNE 2")

    color_pred = np.where(viz_df["y_pred"].to_numpy() == 1, "tab:green", "tab:orange")
    axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=color_pred, s=16, alpha=0.8)
    axes[1].set_title("t-SNE Colored by Predicted Class")
    axes[1].set_xlabel("t-SNE 1")
    axes[1].set_ylabel("t-SNE 2")

    for ax in axes:
        ax.grid(alpha=0.2)

    plt.tight_layout()
    plt.show()

## 12) Grad-CAM Interpretability (CNN Backbones Only)

In [ ]:
if encoder == "vit_b_16":
    display(Markdown("**Grad-CAM skipped:** ViT checkpoint detected (`vit_b_16`)."))
else:
    layer_name, target_layer = last_conv_layer(model)
    if target_layer is None:
        display(Markdown("**Grad-CAM skipped:** no convolution layer found."))
    else:
        print(f"Using Grad-CAM target layer: {layer_name}")

        gradcam_transform = build_eval_transform(crop_size)

        # Prioritize errors, then correct examples.
        pool = pd.concat([
            result_df[result_df["case"] == "FP"],
            result_df[result_df["case"] == "FN"],
            result_df[result_df["case"] == "TP"],
            result_df[result_df["case"] == "TN"],
        ], ignore_index=True).drop_duplicates(subset=["path"])

        if pool.empty:
            pool = result_df.copy()

        selected = pool.head(N_GRADCAM_EXAMPLES)

        n = len(selected)
        fig, axes = plt.subplots(n, 2, figsize=(10, 4 * n))
        axes = np.atleast_2d(axes)

        for i, row in enumerate(selected.itertuples(index=False)):
            rgb = load_rgb(row.path)
            pil = Image.fromarray(rgb)
            tensor = gradcam_transform(pil).unsqueeze(0).to(DEVICE)

            class_idx = int(row.y_pred)  # explain the predicted class
            cam = compute_gradcam(model, tensor, target_layer, class_index=class_idx)
            over = overlay_cam(rgb, cam)

            axes[i, 0].imshow(rgb)
            axes[i, 0].set_title(
                f"Original | true={row.true_label} pred={row.pred_label} p={row.p_good:.2f}"
            )
            axes[i, 0].axis("off")

            axes[i, 1].imshow(over)
            axes[i, 1].set_title("Grad-CAM Overlay")
            axes[i, 1].axis("off")

        plt.tight_layout()
        plt.show()

## 13) Final Doability Decision

Rule:
- Doable = **Yes** iff both conditions hold at `DECISION_THRESHOLD`:
  - `F1 >= PASS_F1`
  - `Specificity >= PASS_SPECIFICITY`

In [ ]:
f1_ok = metrics["f1"] >= PASS_F1
spec_ok = metrics["specificity"] >= PASS_SPECIFICITY
is_doable = bool(f1_ok and spec_ok)

verdict = "YES" if is_doable else "NO"
print(f"Doable verdict: {verdict}")
print(f"F1={metrics['f1']:.4f} (threshold {PASS_F1:.2f})")
print(f"Specificity={metrics['specificity']:.4f} (threshold {PASS_SPECIFICITY:.2f})")

caveats = []

f1_ci = ci_dict.get("f1", (np.nan, np.nan))
spec_ci = ci_dict.get("specificity", (np.nan, np.nan))

if np.isfinite(f1_ci[0]) and f1_ci[0] < PASS_F1:
    caveats.append(
        f"F1 lower CI bound is {f1_ci[0]:.3f}, below pass threshold {PASS_F1:.2f}."
    )
if np.isfinite(spec_ci[0]) and spec_ci[0] < PASS_SPECIFICITY:
    caveats.append(
        f"Specificity lower CI bound is {spec_ci[0]:.3f}, below pass threshold {PASS_SPECIFICITY:.2f}."
    )

n_fp = int((result_df["case"] == "FP").sum())
n_fn = int((result_df["case"] == "FN").sum())
if n_fp > 0:
    caveats.append(f"False positives present: {n_fp} samples.")
if n_fn > 0:
    caveats.append(f"False negatives present: {n_fn} samples.")

counts = result_df["y_true"].value_counts().to_dict()
if 0 in counts and 1 in counts and min(counts.values()) > 0:
    imbalance = max(counts.values()) / min(counts.values())
    if imbalance > 2.0:
        caveats.append(f"Class imbalance ratio is {imbalance:.2f}:1.")

if not caveats:
    caveats = ["No major caveats detected under current checks."]

display(Markdown("### Caveat Summary"))
for c in caveats:
    print(f"- {c}")